In [1]:
# %pip install scipy plotly

In [2]:

from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px

from scipy.spatial import cKDTree


# ==========================================
# 1. Project paths
# ==========================================

PROJECT_DIR = Path.cwd()

INPUT_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "part3_map_ready.csv"
)

TABLE_DIR = PROJECT_DIR / "output" / "tables"
FIGURE_DIR = PROJECT_DIR / "output" / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )



In [3]:

# ==========================================
# 2. Load map-ready dataset
# ==========================================

df = pd.read_csv(INPUT_FILE)

required_columns = [
    "Station",
    "X",
    "Y",
    "Longitude",
    "Latitude",
    "Flow_2019",
    "Flow_2022",
    "Flow_2025",
    "Recovery_Index",
    "Recovery_Group"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

df = df.dropna(
    subset=[
        "X", "Y",
        "Longitude", "Latitude",
        "Recovery_Index"
    ]
).copy()

print("Dataset shape:", df.shape)
print("Unique stations:", df["Station"].nunique())



Dataset shape: (267, 18)
Unique stations: 267


In [4]:

# ==========================================
# 3. Recovery overview
# ==========================================

recovery_summary = (
    df.groupby("Recovery_Group", observed=False)
    .agg(
        Station_Count=("Station", "count"),
        Average_Recovery=("Recovery_Index", "mean"),
        Median_Recovery=("Recovery_Index", "median"),
        Minimum_Recovery=("Recovery_Index", "min"),
        Maximum_Recovery=("Recovery_Index", "max")
    )
    .reset_index()
)

recovery_summary["Percentage"] = (
    recovery_summary["Station_Count"]
    / len(df) * 100
)

recovery_summary.to_csv(
    TABLE_DIR / "part3_recovery_summary.csv",
    index=False
)

print("\nRecovery summary:")
print(recovery_summary.round(2))



Recovery summary:
   Recovery_Group  Station_Count  Average_Recovery  Median_Recovery  \
0  Above baseline             21            120.18           116.38   
1  Below baseline            223             76.06            77.49   
2   Near baseline             23             94.19            93.48   

   Minimum_Recovery  Maximum_Recovery  Percentage  
0            100.26            165.64        7.87  
1             40.36             89.86       83.52  
2             90.15             99.83        8.61  


In [5]:
# print(df[["Longitude", "Latitude"]].describe())

# print("\nMissing coordinates:")
# print(df[["Longitude", "Latitude"]].isna().sum())

# print("\nFirst 5 stations:")
# print(
#     df[["Station", "Longitude", "Latitude"]].head()
# )

In [6]:
# import plotly.express as px

# test_map = px.scatter(
#     df,
#     x="Longitude",
#     y="Latitude",
#     hover_name="Station",
#     title="London Underground Stations"
# )

# test_map.update_layout(
#     width=900,
#     height=650,
#     xaxis_title="Longitude",
#     yaxis_title="Latitude"
# )

# test_map.show()

In [7]:
# ==========================================
# 4. Map 1: Station Recovery Map
# ==========================================

fig1 = px.scatter_map(
    df,
    lat="Latitude",
    lon="Longitude",
    color="Recovery_Index",
    hover_name="Station",
    hover_data={
        "LINES": True,
        "Flow_2019": ":,.0f",
        "Flow_2022": ":,.0f",
        "Flow_2025": ":,.0f",
        "Recovery_Index": ":.1f",
        "Recovery_Group": True,
        "Latitude": False,
        "Longitude": False
    },
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=100,
    zoom=9,
    center={
        "lat": 51.5074,
        "lon": -0.1278
    },
    map_style="open-street-map",
    opacity=0.8,
    title="Where Did Recovery Happen? — London Underground"
)

fig1.update_traces(
    marker=dict(size=10)
)

fig1.update_layout(
    template="plotly_white",
    height=750,
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(
        title="Recovery Index",
        ticksuffix="%"
    )
)

fig1.write_html(
    FIGURE_DIR / "part3_station_recovery_map.html"
)

fig1.show()

In [8]:


# ==========================================
# 5. Spatial neighbourhood analysis
# ==========================================

# Use British National Grid coordinates (metres),
# rather than longitude and latitude, for distance calculations.

coordinates_xy = df[["X", "Y"]].to_numpy()

# Find each station's 8 nearest neighbours.
# The first result is the station itself, so exclude it.

tree = cKDTree(coordinates_xy)

k = min(9, len(df))

distances, indices = tree.query(
    coordinates_xy,
    k=k
)

neighbour_means = []

for i in range(len(df)):

    neighbour_indices = indices[i][1:]

    neighbour_recovery = (
        df.iloc[neighbour_indices]["Recovery_Index"]
    )

    neighbour_means.append(
        neighbour_recovery.mean()
    )

df["Local_Recovery_Mean_8NN"] = neighbour_means

df["Local_Recovery_Gap"] = (
    df["Recovery_Index"]
    - df["Local_Recovery_Mean_8NN"]
)

print("\nLocal neighbourhood analysis:")
print(
    df[
        [
            "Station",
            "Recovery_Index",
            "Local_Recovery_Mean_8NN",
            "Local_Recovery_Gap"
        ]
    ].head()
)



Local neighbourhood analysis:
                Station  Recovery_Index  Local_Recovery_Mean_8NN  \
0            Farringdon      165.641964                73.258281   
1              Richmond      160.886161                73.208005   
2  Tottenham Court Road      144.094407                70.451084   
3             Wimbledon      140.207084                83.907018   
4           Whitechapel      136.144452                76.348366   

   Local_Recovery_Gap  
0           92.383683  
1           87.678156  
2           73.643323  
3           56.300067  
4           59.796085  


In [9]:
# ==========================================
# 6. Map 2: Local Recovery Context
# ==========================================

fig2 = px.scatter_map(
    df,
    lat="Latitude",
    lon="Longitude",
    color="Local_Recovery_Mean_8NN",
    hover_name="Station",
    hover_data={
        "Recovery_Index": ":.1f",
        "Local_Recovery_Mean_8NN": ":.1f",
        "Local_Recovery_Gap": ":.1f",
        "LINES": True,
        "Latitude": False,
        "Longitude": False
    },
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=100,
    zoom=9,
    center={
        "lat": 51.5074,
        "lon": -0.1278
    },
    map_style="open-street-map",
    opacity=0.8,
    title="Local Recovery Context — 8 Nearest Stations"
)

fig2.update_traces(
    marker=dict(size=10)
)

fig2.update_layout(
    template="plotly_white",
    height=750,
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(
        title="Local Average Recovery",
        ticksuffix="%"
    )
)

fig2.write_html(
    FIGURE_DIR / "part3_local_recovery_map.html"
)

fig2.show()

In [10]:

# ==========================================
# 7. Identify local recovery contrasts
# ==========================================

# Stations performing substantially better or worse
# than their surrounding 8 nearest stations.

local_contrasts = df.sort_values(
    "Local_Recovery_Gap",
    ascending=False
)

local_contrasts.to_csv(
    TABLE_DIR / "part3_local_recovery_context.csv",
    index=False
)

print("\nStations outperforming their local neighbourhood:")
print(
    local_contrasts[
        [
            "Station",
            "Recovery_Index",
            "Local_Recovery_Mean_8NN",
            "Local_Recovery_Gap"
        ]
    ].head(10).round(1).to_string(index=False)
)

print("\nStations underperforming their local neighbourhood:")
print(
    local_contrasts[
        [
            "Station",
            "Recovery_Index",
            "Local_Recovery_Mean_8NN",
            "Local_Recovery_Gap"
        ]
    ].tail(10).round(1).to_string(index=False)
)



Stations outperforming their local neighbourhood:
             Station  Recovery_Index  Local_Recovery_Mean_8NN  Local_Recovery_Gap
          Farringdon           165.6                     73.3                92.4
            Richmond           160.9                     73.2                87.7
Tottenham Court Road           144.1                     70.5                73.6
         Whitechapel           136.1                     76.3                59.8
           Wimbledon           140.2                     83.9                56.3
            Moorgate           119.2                     68.6                50.6
         Bond Street           119.6                     69.8                49.9
      Buckhurst Hill           118.4                     70.5                47.9
      Paddington TfL           121.0                     73.3                47.8
     Ealing Broadway           106.8                     65.1                41.7

Stations underperforming their local neighbour

In [11]:
# ==========================================
# 8. Export final spatial dataset
# ==========================================

df.to_csv(
    TABLE_DIR / "part3_spatial_analysis.csv",
    index=False
)

print("\nPart 03 analysis complete.")
print("Tables saved to:", TABLE_DIR)
print("Figures saved to:", FIGURE_DIR)


Part 03 analysis complete.
Tables saved to: /Users/ovo/Desktop/Grad/Internship/Data_for_Humanity_Internship/do-starter-playground/Model/output/tables
Figures saved to: /Users/ovo/Desktop/Grad/Internship/Data_for_Humanity_Internship/do-starter-playground/Model/output/figures
